# Autoformalization Evaluation

In [1]:
import sys, re, spacy
import gale_shapley_algorithm as gsa
sys.path.insert(0, '/home/flopezp/LogicSim')  
import pandas as pd
from logicsim import utils, metrics
from nltk.metrics import distance

dataset_path = '/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{}/{}_all_samples.csv'
model_list = utils.model_list

for elem in model_list:
    model_id = elem[0].split('/')[1]
    # IDEAL USE:
    # metrics.logicsimautoform(model_id)
    # Y QUE SE EVALÚE TODO

In [2]:
dataset_path = '/home/flopezp/Kurosagol/Ongoing/second_round_experiments/{}/{}_all_samples.csv'
model_list = utils.model_list
ex_data = pd.read_csv(dataset_path.format('FOLIO', model_list[0][0].split('/')[1]))
ex_data = ex_data.drop(columns=['prompt_index', 'sample_index', 'prompt_text'])
ex_data.head()

,generated_text
0,<text>\n ∀x (Performs(x) → (Attends(x) ...
1,<text>\n ∀x (Performs(x) → (Attends(x) ...
2,<text>\n ∀x (Performs(x) → (Attends(x) ...
3,<text>\n ∀x (Performs(x) → (Attends(x) ...
4,<text>\n ∀x (Performs(x) → (Attends(x) ...


In [3]:
clean_text = []
none_regex = 0
avg_ans_length = 0
answer_len = len(ex_data['generated_text'].to_list())

for elem in ex_data['generated_text'].to_list():
    avg_ans_length += len(elem)
    text_split = elem.split('<text>')
    # Tal vez no sea search la mejor opción.
    # Podemos evaluar distintas formas de extraer la proposición final usando regex.
    regex_extraction = re.search(r'(<text>)[A-z0-9∀∃\n⊕→¬∧ \t()"á,∨]+(<\/text>)', elem)

    #Second regex
    # regex2_extraction = re.search(r'<text>[\\Śą<>≤A-z0-9:á ∀∧→⊕¬←∨∃↔∈()’\'=≠?.\-,\n"]+<\/text>', elem) 
      
    if regex_extraction != None:
        texto = regex_extraction.group()[6:-7]
        cleaned = re.sub('  ', '', texto)
        clean_text.append(cleaned)
    else:
        none_regex += 1
        clean_text.append(None)

filtered_ans_len = 0
for elem in clean_text:
    if elem != None:
        filtered_ans_len += len(elem)


print(f'Longitud promedio de respuesta: {round(avg_ans_length/answer_len, 4)}')
print(f'Valores totales: {answer_len}')
print(f'Valores mal generados: {none_regex}. Porcentaje: {round(none_regex/answer_len, 4)*100}%')
print(f'Longitud promedio de respuesta filtrada: {round(filtered_ans_len/(answer_len - none_regex), 4)}')

Longitud promedio de respuesta: 5249.9202
Valores totales: 1015
Valores mal generados: 178. Porcentaje: 17.54%
Longitud promedio de respuesta filtrada: 194.963


In [4]:
folio_val = pd.read_json(r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_validation.jsonl', lines = True)
folio_premises = folio_val['premises-FOL'].to_list()

# ----------------------------------------------------------------------------------------------------------
# Con esto ya podemos medir la primeras dos partes de la métrica: PARSING & CARDINALITY_EQUALITY
# ----------------------------------------------------------------------------------------------------------

llm_parse = False
ds_parse = False
cardinal_list = []

llm_parse_count, dataset_parse_count = 0,0
parsing_pairs = 0

for i in range(int(len(clean_text)/5)):
    ds_value = folio_premises[i]
    nones = 0
    for j in range(5):
        current_index = i*5 + j
        print('-'*50)
        print('\t Evaluating instance ', current_index)
        print('-'*50)
        if clean_text[current_index] == None:
            nones += 1
        else:
            ds_parse = metrics.lark_based_parsability(ds_value, parser = metrics.parser)
            llm_parse = metrics.lark_based_parsability(clean_text[current_index], parser = metrics.parser)
            print('-'*20, 'Parsing', '-'*20)
            print(f'DS Parses: {ds_parse}')
            print(f'LLM Autoform Parses: {llm_parse}')
            if llm_parse:
                llm_parse_count += 1
            if ds_parse:
                dataset_parse_count += 1

            if llm_parse and ds_parse:
                parsing_pairs += 1
                try:
                    print('-'*20, 'Cardinality', '-'*20)
                    cardinal_equality = metrics.verify_cardinality(ds_value, clean_text[current_index])
                    print(f'Cardinality Equality: {cardinal_equality[0]}')
                    print(f'Constant Errors: {cardinal_equality[1]}')
                    print(f'Predicate Errors: {cardinal_equality[2]}')
                    if cardinal_equality[0] == True:
                        cardinal_list.append((i, current_index))
                except:
                    print('-'*20, 'Cardinality', '-'*20)
                    print(f'Cardinality Equality: False')
                    print(f'DS value: {ds_value}')
                    print(f'LLM value: {clean_text[current_index]}')

    print('-'*25)


print('='*50)
print('\t \t Final Results')
print('='*50)
print('\t \t --- PARSING ---')
print(f'LLM Parse Rate: {round(llm_parse_count/len(clean_text), 4)*100}%')
print(f'Dataset Parse Rate: {round(dataset_parse_count/len(clean_text), 4)*100}%')
print(f'Cantidad de instancias que parsean: {parsing_pairs}')
print('\t \t --- CARDINALITY ---')
print(f'Cardinality Equivalence Rate: {round(len(cardinal_list)/len(clean_text), 4)*100}%')
print(f'Cardinal equal values: {len(cardinal_list)}')

--------------------------------------------------
	 Evaluating instance  0
--------------------------------------------------
-------------------- Parsing --------------------
DS Parses: True
LLM Autoform Parses: True
-------------------- Cardinality --------------------
Cardinality Equality: False
Constant Errors: 7
Predicate Errors: 4
--------------------------------------------------
	 Evaluating instance  1
--------------------------------------------------
-------------------- Parsing --------------------
DS Parses: True
LLM Autoform Parses: True
-------------------- Cardinality --------------------
Cardinality Equality: False
Constant Errors: 7
Predicate Errors: 4
--------------------------------------------------
	 Evaluating instance  2
--------------------------------------------------
-------------------- Parsing --------------------
DS Parses: True
LLM Autoform Parses: True
-------------------- Cardinality --------------------
Cardinality Equality: False
Constant Errors: 7


In [ ]:
# Edit distance only isomorphism creation. HAS PROBLEMS.
for elem in cardinal_list:
    print('='* 40)
    print(f'\t Índice: {elem}')
    gold = folio_premises[elem[0]]
    llm = clean_text[elem[1]]
    query = folio_val['conclusion-FOL'].iloc[elem[0]]
    gold_ag, llm_ag, query_ag = metrics.name_agnostic_transformation(gold, llm, query)

    gold_p9 = [utils.clean_for_lark(value) for value in gold_ag.split('\n')]
    while '' in gold_p9:
            gold_p9.remove('')

    llm_p9 = [utils.clean_for_lark(value) for value in llm_ag.split('\n')]
    while '' in llm_p9:
            llm_p9.remove('')

    query_p9 = [utils.clean_for_lark(value) for value in query_ag.split('\n')]
    while '' in query_p9:
            query_p9.remove('')
    #print(f'Gold Name Agnostic: \n {gold_ag}')
    print('-'* 15)
    print(f'Gold P9:\n {gold_p9}')
    print('-'*15)
    #print(f'LLM Name Agnostic: {llm_ag}')
    print(f'LLM P9:\n {llm_p9}')
    print('-'*15)
    #print(f'Queries Name Agnostic: \n {query_ag}')
    print(f'Query P9:\n {query_p9}')


	 Índice: (9, 45)
[[{'easternwildturkey': 'easternwildturkey', 'gouldswildturkey': 'gouldswildturkey', 'merriamswildturkey': 'merriamswildturkey', 'ocellatedwildturkey': 'ocellatedwildturkey', 'osceolawildturkey': 'osceolawildturkey', 'riograndewildturkey': 'riograndewildturkey', 'wildturkey': 'wildturkey'}, ['pred1a0', 'pred1a1', 'pred1a2', 'pred1a3', 'pred1a4', 'pred1a5', 'pred1a6']]]
[{'tom': 'tom'}, ['const0']]
---------------
Gold P9:
 ['∀x (pred1a6(x) → (pred1a0(x) ∨ pred1a4(x) ∨ pred1a1(x) ∨ pred1a2(x) ∨ pred1a5(x) ∨ pred1a3(x)))', '¬(pred1a0(const0))', '¬(pred1a4(const0))', '¬(pred1a1(const0))', '¬(pred1a2(const0) ∨ pred1a5(const0))', 'pred1a6(const0)']
---------------
LLM P9:
 ['∀x (pred1a6(x) → (pred1a0(x) ∨ pred1a4(x) ∨ pred1a1(x) ∨ pred1a2(x) ∨ pred1a5(x) ∨ pred1a3(x)))', '¬pred1a0(const0)', '¬pred1a4(const0)', '¬pred1a1(const0)', '¬pred1a2(const0) ∧ ¬pred1a5(const0)', 'pred1a6(const0)']
---------------
Query P9:
 ['pred1a3(const0)']
	 Índice: (10, 50)
[[{'easternwildturkey

In [5]:
# Vamos a tratar de reworkear la función del isomorfismo y la validación.

ds_value = folio_premises[177]
llm_value = clean_text[888]
query = folio_val['conclusion-FOL'].iloc[177]

In [10]:
def gs_weights(base, objective):
    """
    Dadas dos listas de predicados/constantes, se obtiene la distancia de edición de los valores de la lista base
    con respecto a los valores de la lista objetivo. Formatea la lista para que se pueda usar con la paqutería de
    GSA.

    base = list
    objective = list
    """
    weights = {elem:[] for elem in base}
    for i in range(len(base)):
        current_base = base[i]
        current_base_distances = []

        for j in range(len(objective)):
            current_obj = objective[j]
            dist = distance.edit_distance(current_base, current_obj) # ELEMENTO A MODIFICAR
            current_base_distances.append((dist, current_obj))
            current_base_distances = sorted(current_base_distances)

        current_base_distances = [_[1] for _ in current_base_distances]
        weights[current_base] = current_base_distances
    return weights

def isomorphism(dataset_value, llm_value):
    """
    Genera un isomorfismo entre los valores del conjunto de datos y la autoformalización del modelo de lenguaje.

    dataset_value = str ; 
    llm_value = str ; 

    return values:
    
    isomorfismo = dict ; Isomorphism based on edit distance and gale-shapley
    mixed_iso = dict ; Isomorphism with modified structure for further processing.
    """
    # We extract predicate and constant info
    ds_preds, ds_consts, _, _ = metrics.extract_info(dataset_value, False)
    llm_preds, llm_consts, _, _ = metrics.extract_info(llm_value, False)

    # We separate predicates based on their arity
    ds_arity = metrics.get_arity_list(ds_preds) 
    llm_arity = metrics.get_arity_list(llm_preds)
    gold_arities = list(set(a[1] for a in ds_arity[0]))
    llm_arities = list(set(a[1] for a in llm_arity[0]))

    # Normalization before obtaining weightsa
    ds_predicate_list, llm_predicate_list = [], []
    ds_constants_list = [elem.lower() for elem in ds_consts]
    llm_constants_list = [elem.lower() for elem in llm_consts]
    
    for elem in gold_arities:
        current_pred = [_[0].lower() if _[1] == elem else None for _ in ds_arity[0]]
        while None in current_pred:
            current_pred.remove(None)
        ds_predicate_list.append(current_pred)

    for elem in llm_arities:
        current_llm_pred = [_[0].lower() if _[1] == elem else None for _ in llm_arity[0]]
        while None in current_llm_pred:
            current_llm_pred.remove(None)
        llm_predicate_list.append(current_llm_pred)

    # ds_predicate_list, llm_predicate_list, ds_constants_list, llm_constants_list -> All values needed and sorted
    # for weight measuring and isomorphism creation.
    
    isomorfismo = []

    # Constant matching
    # IF WE WANT TO MODIFY HOW WE MEASURE SIMILARITY BETWEEN SAME-ARITY VALUES WE HAVE TO MODIFY THE FUNCTION gs_weights(a, b)
    gold_const_weight, llm_const_weight = gs_weights(ds_constants_list, llm_constants_list), gs_weights(llm_constants_list, ds_constants_list)
    constant_iso = gsa.create_matching(gold_const_weight, llm_const_weight).matches
    isomorfismo.append(constant_iso)

    arity = len(ds_predicate_list)
    
    # Gale-Shapley
    for i in range(arity):
        gold_current_arity = ds_predicate_list[i]
        llm_current_arity = llm_predicate_list[i]
        gold_pred_weight, llm_pred_weight = gs_weights(gold_current_arity, llm_current_arity), gs_weights(llm_current_arity, gold_current_arity)
        current_arity_iso = gsa.create_matching(gold_pred_weight, llm_pred_weight).matches
        isomorfismo.append(current_arity_iso)
    
    #print('-'*15, 'Isomorfismo', '-'*15) 
    name_agnostic = []
    for i in range(len(isomorfismo)):
        if i == 0:
            #print(f'Constantes: \n {isomorfismo[i]}') # Solo si queremos ver el isomorfismo antes
            const_name_agnostic = [f'const{j}' for j in range(len(isomorfismo[i]))]
            name_agnostic.append(const_name_agnostic)
        else:
            #print(f'Predicados de aridad {i}: \n {isomorfismo[i]}') # Solo si queremos ver el isomorfismo antes
            pred_name_agnostic = [f'pred{i}a{j}' for j in range(len(isomorfismo[i]))]
            name_agnostic.append(pred_name_agnostic)
    
    #print(name_agnostic)
    mixed_iso = {}
    for i in range(len(isomorfismo)):
        iso_current = isomorfismo[i]
        name_current = name_agnostic[i]

        iso_keys = list(iso_current.keys())
        iso_values = list(iso_current.values())
        for j in range(len(iso_current)):
            mixed_iso[name_current[j]] = [iso_keys[j], iso_values[j]]

    print(isomorfismo)
    print('-'*15)
    print(mixed_iso)

    return isomorfismo, mixed_iso


def name_switch(string, mixed_iso):
    """
        string = str ; El texto a anonimizar, permite ser una serie de premisas.
        mixed_iso = dict ; El isomorfismo mixto obtenido del isomophism(a, b).
    """
    split_value = string.split('\n')
    while '' in split_value:
        split_value.remove('')

    split_value = [elem.lower() for elem in split_value]
    modded = []

    # Me cago EN PUTO CRISTO
    # ¿CÓMO QUE CÚBICO CABRÓN?
    # Tenemos que hacer uso de nuestro buen amigo chat.
    for sentence in split_value:
        modded_sentence = sentence
        for elem in mixed_iso:
            changes = mixed_iso[elem]
            for value in changes:
                explicit_search = re.search(rf'\b{value}\b', modded_sentence)
                if explicit_search != None:
                    modded_sentence = re.sub(value, elem, modded_sentence)
        modded.append(utils.clean_for_lark(modded_sentence))
    return modded


In [ ]:
# FUNCIONA LA PUTA VERGA FUNCIONAAAAAAAAAAAAAAAAAAA
iso, mixed = isomorphism(ds_value, llm_value)

print(ds_value)
print(name_switch(ds_value, mixed))
print('-'*5)
print(llm_value)
print(name_switch(llm_value, mixed))
print('-'*5)
print(query)
print(name_switch(query, mixed))


[{'olympics': 'max', 'unitedstates': 'summerolympicgames', 'tokyo': 'tokyo'}, {'lastsummerolympics': 'lastolympicevent', 'sportingevent': 'olympicevent'}, {'mostmedals': 'usmedals'}]
---------------
{'const0': ['olympics', 'max'], 'const1': ['unitedstates', 'summerolympicgames'], 'const2': ['tokyo', 'tokyo'], 'pred1a0': ['lastsummerolympics', 'lastolympicevent'], 'pred1a1': ['sportingevent', 'olympicevent'], 'pred2a0': ['mostmedals', 'usmedals']}
SportingEvent(olympics)
LastSummerOlympics(tokyo)
MostMedals(unitedStates, tokyo)
['pred1a1(const0)', 'pred1a0(const2)', 'pred2a0(const1, const2)']
-----

OlympicEvent(SummerOlympicGames)
LastOlympicEvent(Tokyo)
USMedals(Tokyo, max)

['pred1a1(const1)', 'pred1a0(const2)', 'pred2a0(const2, const0)']
-----
∃x (LastSummerOlympics(x) ∧ MostMedals(unitedStates, x))
['∃x (pred1a0(x) ∧ pred2a0(const1,  x))']
